In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
!pip install -U sentence-transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
!pip install clip-by-openai

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import pandas as pd
import numpy as np
import random, time
import matplotlib.pylab as plt
import matplotlib as mpl
from matplotlib.collections import LineCollection
from sklearn.metrics import accuracy_score, precision_recall_fscore_support,confusion_matrix
from tqdm.notebook import tqdm, trange
import json
from PIL import Image

import torch
from torch.autograd import Variable
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from sklearn.metrics import precision_recall_curve
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from matplotlib import pyplot



from torchvision import models
from torchvision.transforms import ToTensor
import os
from tqdm.notebook import tqdm
from collections import Counter

In [ ]:
from sentence_transformers import SentenceTransformer
import xgboost

In [ ]:
import clip

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
train_data = pd.read_json("/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/train.jsonl", lines=True)
val_seen_data = pd.read_json("/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/dev_seen.jsonl", lines=True)
val_unseen_data = pd.read_json("/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/dev_unseen.jsonl", lines=True)
test_seen_data = pd.read_json("/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/test_seen.jsonl", lines=True)
test_unseen_data = pd.read_json("/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/test_unseen.jsonl", lines=True)

In [ ]:
type(train_data)

pandas.core.frame.DataFrame

In [ ]:
#train_data = train_data.head(20)

In [ ]:
train_data.head()

,id,img,label,text
0,42953,img/42953.png,0,its their character not their color that matters
1,23058,img/23058.png,0,don't be afraid to love again everyone is not ...
2,13894,img/13894.png,0,putting bows on your pet
3,37408,img/37408.png,0,i love everything and everybody! except for sq...
4,82403,img/82403.png,0,"everybody loves chocolate chip cookies, even h..."


## Generate text and image embeddings

In [ ]:
class Embeddings:
    def __init__(self, dataframe, text_embedding_model, image_embedding_model):
        self.dataframe = dataframe
        self.text_embedding_model = text_embedding_model
        self.image_embedding_model = image_embedding_model
        self.images_path = '/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/'

    # Text
    def generate_and_add_text_embeddings_to_df(self):
        sentences = list(self.dataframe['text'])
        embeddings = self.text_embedding_model.encode(sentences)

        self.dataframe['text_embeddings'] = ""
        for i in range(self.dataframe.shape[0]):
            self.dataframe.at[i, 'text_embeddings'] = embeddings[i]


    # Image
    def convert_image_to_embeddings(self, image):     # Helper function
        with torch.no_grad():                                                       
            feature_extractor = torch.nn.Sequential(*list(self.image_embedding_model.children())[:-1]) # Removed last layer of Resnext
            image_tensor = ToTensor()(image)[:3].unsqueeze(0)   # Take just the first 3 channels and unsqueeze                          
            features = feature_extractor(torch.tensor(image_tensor))                
            return features.squeeze()[:-1].tolist()  # Not sure why last element is dropped

    def generate_and_add_image_embeddings_to_df(self):
        self.image_embedding_model.eval()

        img_names_list = list(self.dataframe['img'])
        img_tensors = []
        for name in tqdm(img_names_list):
            image = Image.open(self.images_path + name)
            image_tensor = self.convert_image_to_embeddings(image = image)
            img_tensors.append(image_tensor)
        print('Number of images encoded:', len(img_tensors))

        self.dataframe['img_embeddings'] = ""
        for i in range(self.dataframe.shape[0]):
            self.dataframe.at[i, 'img_embeddings'] = img_tensors[i]




In [ ]:
text_embedding_model = SentenceTransformer('sentence-transformers/msmarco-roberta-base-v2')
image_embedding_model = models.resnext50_32x4d(pretrained=True)

Downloading:   0%|          | 0.00/391 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/190 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/3.70k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/678 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/122 [00:00<?, ?B/s]

...

Downloading:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/798k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/229 [00:00<?, ?B/s]

/usr/local/lib/python3.8/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.8/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNeXt50_32X4D_Weights.IMAGENET1K_V1`. You can also use `weights=ResNeXt50_32X4D_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnext50_32x4d-7cdf4587.pth" to /root/.cache/torch/hub/checkpoints/resnext50_32x4d-7cdf4587.pth


  0%|          | 0.00/95.8M [00:00<?, ?B/s]

In [ ]:
train_data_embeddings = Embeddings(dataframe = train_data, text_embedding_model = text_embedding_model, image_embedding_model = image_embedding_model)
train_data_embeddings.generate_and_add_text_embeddings_to_df()
train_data_embeddings.generate_and_add_image_embeddings_to_df()

NameError: ignored

In [ ]:
val_seen_data_embeddings = Embeddings(dataframe = val_seen_data, text_embedding_model = text_embedding_model, image_embedding_model = image_embedding_model)
# val_seen_data_embeddings.generate_and_add_text_embeddings_to_df()
val_seen_data_embeddings.generate_and_add_image_embeddings_to_df()
val_seen_data.to_csv('/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/' + 'val_seen_data' + '_with_image_and_text_embeddings.csv')

In [ ]:
val_unseen_data_embeddings = Embeddings(dataframe = val_unseen_data, text_embedding_model = text_embedding_model, image_embedding_model = image_embedding_model)
val_unseen_data_embeddings.generate_and_add_text_embeddings_to_df()
val_unseen_data_embeddings.generate_and_add_image_embeddings_to_df()
val_unseen_data.to_csv('/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/' + 'val_unseen_data' + '_with_image_and_text_embeddings.csv')

In [ ]:
test_seen_data_embeddings = Embeddings(dataframe = test_seen_data, text_embedding_model = text_embedding_model, image_embedding_model = image_embedding_model)
test_seen_data_embeddings.generate_and_add_text_embeddings_to_df()
test_seen_data_embeddings.generate_and_add_image_embeddings_to_df()
test_seen_data.to_csv('/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/' + 'test_seen_data' + '_with_image_and_text_embeddings.csv')

  0%|          | 0/1000 [00:00<?, ?it/s]<ipython-input-8-8bf44bf0a798>:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  features = feature_extractor(torch.tensor(image_tensor))
100%|██████████| 1000/1000 [30:38<00:00,  1.84s/it]


Number of images encoded: 1000


In [ ]:
test_unseen_data_embeddings = Embeddings(dataframe = test_unseen_data, text_embedding_model = text_embedding_model, image_embedding_model = image_embedding_model)
test_unseen_data_embeddings.generate_and_add_text_embeddings_to_df()
test_unseen_data_embeddings.generate_and_add_image_embeddings_to_df()
test_unseen_data.to_csv('/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/' + 'test_unseen_data' + '_with_image_and_text_embeddings.csv')

  0%|          | 0/2000 [00:00<?, ?it/s]<ipython-input-13-5e8960b66f83>:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  features = feature_extractor(torch.tensor(image_tensor))
  0%|          | 0/2000 [00:00<?, ?it/s]


RuntimeError: ignored

In [ ]:
# Img embeddings = 2047
# Text embeddings = 768
# 2047+768 = 2815

In [ ]:
train_data = pd.read_csv('/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/train_data_with_image_and_text_embeddings.csv')

NameError: ignored

In [ ]:
test_seen = pd.read_csv('/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/test_seen_data_with_image_and_text_embeddings.csv')

In [ ]:
len(train_data['text_embeddings'][0])

12480

In [ ]:
len(train_data['img_embeddings'][0])

41769

In [ ]:
train_data

,Unnamed: 0,id,img,label,text,img_embeddings,text_embeddings
0,0,42953,img/42953.png,0,its their character not their color that matters,"[0.18145230412483215, 0.4469326138496399, 0.78...",[ 1.63657933e-01 2.23474577e-01 5.36924183e-...
1,1,23058,img/23058.png,0,don't be afraid to love again everyone is not ...,"[0.07307668030261993, 0.14269915223121643, 0.9...",[-4.07327086e-01 -2.72272080e-01 6.30129278e-...
2,2,13894,img/13894.png,0,putting bows on your pet,"[0.18135572969913483, 0.10037024319171906, 0.2...",[-1.23310971e+00 6.16022289e-01 -1.44316971e-...
3,3,37408,img/37408.png,0,i love everything and everybody! except for sq...,"[0.0826779156923294, 0.21045134961605072, 0.27...",[ 9.89955366e-02 -8.89315188e-01 -5.06708384e-...
4,4,82403,img/82403.png,0,"everybody loves chocolate chip cookies, even h...","[0.32568907737731934, 0.21178077161312103, 0.5...",[-0.10146231 -0.56808263 -0.8051793 0.303080...
...,...,...,...,...,...,...,...
8495,8495,10423,img/10423.png,1,nobody wants to hang auschwitz me,"[0.2051185667514801, 0.34409821033477783, 0.10...",[ 2.70995647e-01 -8.30023766e-01 -3.98317650e-...
8496,8496,98203,img/98203.png,1,when god grants you a child after 20 years of ...,"[0.2767680585384369, 0.04327409714460373, 0.53...",[-1.62766054e-01 1.66897774e-01 -4.25342828e-...
8497,8497,36947,img/36947.png,1,gays on social media: equality! body positivit...,"[0.5633282661437988, 0.18848328292369843, 0.35...",[ 5.73261619e-01 -5.17796636e-01 4.69983369e-...
8498,8498,16492,img/16492.png,1,having a bad day? you could be a siamese twin ...,"[0.3435825705528259, 0.12689658999443054, 0.36...",[ 6.82777464e-02 -2.52371937e-01 2.05102861e-...


In [ ]:
# train_data.to_csv('/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/' + 'train_data' + '_with_image_and_text_embeddings.csv')

In [ ]:
# train_data = pd.read_csv(r'/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/' + 'train_data' + '_with_image_and_text_embeddings.csv')

In [ ]:
type(train_data['img_embeddings'][0])

In [ ]:
type(train_data['text_embeddings'][3])

In [ ]:
train_data.head()

In [ ]:
val_seen_data.head()

In [ ]:
def convert_strings_list_to_floats_list(arr):
    arr = arr.strip('][ ')
    arr = arr.split(' ')
    temp = []
    for s in arr:
        if s != '' and s != ' ' :
            temp.append(s.strip('\n,'))
    ans = list(map(float, temp))
    return ans

In [ ]:
text_embedding_len = 768
image_embedding_len = 2047
X_train = np.empty((0, text_embedding_len + image_embedding_len), float)
Y_train = np.empty((0, ), int)

# c = 0
for index, row in tqdm(train_data.iterrows()):
    text_emb = row['text_embeddings']
    img_emb = row['img_embeddings']
    if isinstance(text_emb, str):
        text_emb = convert_strings_list_to_floats_list(text_emb)
        img_emb = convert_strings_list_to_floats_list(img_emb)
    # print(len(text_emb))
    # print(len(img_emb))
    merged_emb = np.concatenate( [text_emb, img_emb]  )
    X_train = np.append(X_train, [merged_emb], axis = 0)
    Y_train = np.append(Y_train, [row['label']], axis = 0)

print(X_train.shape)
print(Y_train.shape)

8500it [04:07, 34.31it/s]

(8500, 2815)
(8500,)


In [ ]:
text_embedding_len = 768
image_embedding_len = 2047
X_train = np.empty((0, text_embedding_len + image_embedding_len), float)
Y_train = np.empty((0, ), int)

# c = 0
for index, row in tqdm(train_data.iterrows()):
    text_emb = row['text_embeddings']
    img_emb = row['img_embeddings']
    if isinstance(text_emb, str):
        text_emb = convert_strings_list_to_floats_list(text_emb)
        img_emb = convert_strings_list_to_floats_list(img_emb)
    # print(len(text_emb))
    # print(len(img_emb))
    merged_emb = np.concatenate( [text_emb, img_emb]  )
    X_train = np.append(X_train, [merged_emb], axis = 0)
    Y_train = np.append(Y_train, [row['label']], axis = 0)

print(X_train.shape)
print(Y_train.shape)

In [ ]:
text_list = list(train_data['text'])
max([len(text.split(' ')) for text in text_list])

70

In [ ]:
model, preprocess = clip.load("ViT-B/32", device=device)

100%|██████████████████████| 353976522/353976522 [00:01<00:00, 290264770.21it/s]


In [ ]:
def generate_CLIP_embeddings(dataframe):
    path = '/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/'
    text_list = list(dataframe['text'])
    img_names_list = list(dataframe['img'])
    concatenated_embedding_list = []

    with torch.no_grad():
        for i, (text, img_name) in tqdm(enumerate(zip(text_list, img_names_list))):
            try:
                # words = text.split(' ')
                # if len(words) > 77:
                #     text_embedding = np.zeros((1, 512))
                #     parts = len(words)/77
                #     for i in range(parts):
                #         text_part = ''.join(words[i*77:(i+1)*77])
                #         text_part = clip.tokenize([text_part]).to(device)
                #         text_embedding_part = model.encode_text(text_part)
                #         text_embedding += text_embedding_part
                #     text_embedding /= (parts + 1)
                # else:
                text = clip.tokenize([text]).to(device)
                text_embedding = model.encode_text(text)

                # print('text.shape=', text.shape) 
                img = preprocess(Image.open(path + img_name)).unsqueeze(0).to(device)
                # print('img.shape=', img.shape)
                # print('text_embedding.shape=', text_embedding.shape)
                img_embedding = model.encode_image(img)
                # print('img_embedding.shape=', text_embedding.shape)
                concatenated_embedding = text_embedding[0].tolist()
                concatenated_embedding.extend(img_embedding[0].tolist())
                # print('concatenated_embedding.shape=', len(concatenated_embedding))
                concatenated_embedding_list.append(concatenated_embedding)
            except:
                print(f"{i} didn't work! Gave error :((")

    return np.array(concatenated_embedding_list)


In [ ]:
x_train_clip = generate_CLIP_embeddings(train_data)

0it [00:00, ?it/s]

3527 didn't work! Gave error :((
3954 didn't work! Gave error :((
7581 didn't work! Gave error :((
8242 didn't work! Gave error :((


In [ ]:
print(x_train_clip.shape)

NameError: ignored

In [ ]:
#Runs out of memory, needs to be done in batch. 

#product of embeddings
text_embedding_len = 768
image_embedding_len = 2047
x_train_mul = []
y_train_mul = []
# c = 0
for index, row in tqdm(train_data.iterrows()):
    text_emb = row['text_embeddings']
    img_emb = row['img_embeddings']
    if isinstance(text_emb, str):
        text_emb = torch.tensor(convert_strings_list_to_floats_list(text_emb), dtype = torch.float32).to(device)
        img_emb = torch.tensor(convert_strings_list_to_floats_list(img_emb), dtype = torch.float32).to(device)
    merged_emb = text_emb.reshape(1, 768)*((img_emb).reshape((2047,1)))
    merged_emb = merged_emb.reshape(-1)
    x_train_mul.append(merged_emb)
    y_train_mul.append([row['label']])

print(x_train_mul.shape)
print(y_train_mul.shape)

In [ ]:
text_embedding_len = 768
image_embedding_len = 2047
X_test = np.empty((0, text_embedding_len + image_embedding_len), float)
Y_test = np.empty((0, ), int)

# c = 0
for index, row in tqdm(test_seen.iterrows()):
    text_emb = row['text_embeddings']
    img_emb = row['img_embeddings']
    if isinstance(text_emb, str):
        text_emb = convert_strings_list_to_floats_list(text_emb)
        img_emb = convert_strings_list_to_floats_list(img_emb)
    # print(len(text_emb))
    # print(len(img_emb))
    merged_emb = np.concatenate( [text_emb, img_emb]  )
    X_test = np.append(X_test, [merged_emb], axis = 0)
    Y_test = np.append(Y_test, [row['label']], axis = 0)

print(X_test.shape)
print(Y_test.shape)

1000it [00:06, 161.57it/s]

(1000, 2815)
(1000,)


In [ ]:

from sklearn.metrics import auc, accuracy_score, confusion_matrix, mean_squared_error

xgb_model = xgboost.XGBClassifier(objective="binary:logistic", random_state=42)

xgb_model.fit(X_train, Y_train)

Y_pred = xgb_model.predict(X_train)

print(confusion_matrix(Y_train, Y_pred))
print(accuracy_score(Y_train, Y_pred))

In [ ]:
X_train.shape

(8500, 2815)

In [ ]:
test_pred = xgb_model.predict(X_test)
print(accuracy_score(Y_test, test_pred))

0.566


In [ ]:
device = torch.device('cuda')

In [ ]:
np.save('/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/x_train', X_train)

In [ ]:
np.save('/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/x_test', X_test)
np.save('/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/y_train', Y_train)
np.save('/content/gdrive/Shareddrives/ESE_546_Project/hateful_memes/y_test', Y_test)

# Late Fusion

In [ ]:
class MyDataset():
  def __init__(self,x_train,y_train):
    self.x_train = torch.tensor(x_train, dtype = torch.float32)
    self.y_train = torch.tensor(y_train, dtype = torch.float32)

  def __len__(self):
    return len(self.y_train)

  def __getitem__(self, idx):
    return self.x_train[idx], self.y_train[idx]

In [ ]:
X_train = np.load('x_train.npy')
Y_train = np.load('y_train.npy')
Y_test = np.load('y_test.npy')
X_test = np.load('x_test.npy')

In [ ]:
train_data = MyDataset(X_train,Y_train)
test_data = MyDataset(X_test, Y_test)

In [ ]:
device = torch.device('cuda')

In [ ]:
train_loader = DataLoader(train_data, batch_size=128,
                                          shuffle=True, num_workers=0)
test_loader = DataLoader(test_data, batch_size=128,
                                          shuffle=True, num_workers=0)

NameError: ignored

In [ ]:
from torch.nn.modules.activation import Sigmoid
class FNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO
        self.function = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2815,500),
            nn.ReLU(),
            nn.Linear(500, 10),
            nn.ReLU(),
            nn.Linear(10,1),
            nn.Sigmoid()  
        )
        # END TODO

    def forward(self, x):
        # TODO
        outputs = self.function(x)
        # END TODO
        return outputs

[37.583,
 49.0,
 47.971,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0,
 49.0]

In [ ]:
%%time
# Sending the data to device (CPU or GPU)
# TODO

fnn = FNN().to(device)#.to(device)

criterion = nn.BCELoss().to(device)
# END TODO 
optimizer = optim.Adam(fnn.parameters(), lr=1e-4) #lr - learning step
epoch = 50

acc_LIST_FNN = []
loss_LIST_FNN = []


# Train the FNN
for epoch in range(epoch):
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels in train_loader:
        labels = labels.type(torch.LongTensor) # Cast to Long
        inputs, labels = inputs.to(device), labels.to(device)
        # TODO: Complete the body of this for-loop
        #inputs = inputs.reshape(-1, 3*32*32)
        outputs = fnn.forward(inputs)
        loss = criterion(outputs.flatten(), labels.float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss = loss.item()
        predictions = (outputs.flatten() > 0.5).int()
        correct += (predictions == labels).sum()
        total += len(labels)
  # Calculate Training Acc
    acc_LIST_FNN.append((100*correct / total).item())
    loss_LIST_FNN.append(running_loss/ len(train_loader)) # get the avg loss for each epoch
  # END TODO 
  # print statistics
    print("The loss for Epoch {} is: {}, Accuracy = {}".format(epoch, running_loss/len(train_loader), 100*correct/total))

In [ ]:
total = 0
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        labels = labels.type(torch.LongTensor) # Cast to Float
        images, labels = images.to(device), labels.to(device)
        # TODO: complete each iteration of the for-loop
        outputs = fnn(images) # shape: torch.Size([10000, 10])
        predictions = (outputs.flatten() > 0.5).int()
        correct += (predictions == labels).sum()
        total += len(labels)
test_acc_FNN = 100*correct/total
# TODO END
print('Test Accuracy: ' + str(test_acc_FNN))